# DATA PRE PROCESSING

In [ ]:
import pandas as pd
df = pd.read_csv("AMJATH.txt",header=None,on_bad_lines='skip',encoding='utf8')
df

In [ ]:
df=df.drop(0)
df=df.drop(1)
df.drop([2, 3], axis=1, inplace=True)
df

In [ ]:
df.columns=['Date','Chat']

In [ ]:
Message=df['Chat'].str.split('-',n=1,expand=True)

In [ ]:
df['Time']=Message[0]

In [ ]:
Message1=Message[1].str.split(':',n=1,expand=True)

In [ ]:
df['Name']=Message1[0]
df['Chat']=Message1[1]

In [ ]:
df=df[['Date','Time','Name','Chat']]
df

# SENTIMENTAL ANALYSIS

In [ ]:
data=df
data

In [ ]:
 def sentimentalAnalysis(data,columnname):
        from nltk.sentiment.vader import SentimentIntensityAnalyzer
        data.dropna(inplace=True)
        sid = SentimentIntensityAnalyzer()
        data = data.dropna(subset=[columnname])
        data['scores'] = data[columnname].apply(lambda commentText: sid.polarity_scores(commentText))
        data['compound']  = data['scores'].apply(lambda score_dict: score_dict['compound'])
        data['Negtive']  = data['scores'].apply(lambda score_dict: score_dict['neg'])
        data['Postive']  = data['scores'].apply(lambda score_dict: score_dict['pos'])
        data['Neutral']  = data['scores'].apply(lambda score_dict: score_dict['neu'])
        data['comp_score'] = data['compound'].apply(lambda c: 'pos' if c >=0 else 'neg')
        posneg=pd.DataFrame(data['comp_score'].value_counts())
        return posneg,data

In [ ]:
print(data.columns)

In [ ]:
pos,data_Senti=sentimentalAnalysis(data,columnname='Chat')

In [ ]:
data_Senti

In [ ]:
pos

# TOPIC MODELLING

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf=TfidfVectorizer(max_df=0.95,min_df=2,stop_words='english')
dtm=tfidf.fit_transform(df["Chat"])

In [ ]:
from sklearn.decomposition import NMF
nmf_model=NMF(n_components=5,random_state=42)
nmf_model.fit(dtm)

In [ ]:
for index, topic in enumerate(nmf_model.components_):
    feature_names = tfidf.get_feature_names_out()
    results = [feature_names[i] for i in topic.argsort()[-10:]]
    print(results)

In [ ]:
topic_results=nmf_model.transform(dtm)
df["Topic"]=topic_results.argmax(axis=1)
df

In [ ]:
df["Topic"].value_counts()